# TCC — treino e avaliação no Google Colab (GPU)

Análise de sobrevivência em câncer de mama (SEER 2000–2022): Cox, Cox com splines, RSF, Gradient Boosting, DeepSurv e DeepHit.

**Rode as células em ordem:**
- Preparação (células 2 a 6) e conferência do CSV (célula 8).
- **Passo 2** — estabilidade dos modelos profundos na **validação** (o teste não é usado).
- **Passo 3** — **avaliação final**, que abre o **conjunto de teste uma única vez**.

**Antes de começar:**
1. O código precisa estar no GitHub (`CaetanoPorto/tcc-sobrevivencia-cancer-mama`). Se o repositório for privado, veja a observação na célula de clone.
2. O arquivo `breast_cancer.csv` precisa estar no seu Google Drive (ex.: `MyDrive/TCC/breast_cancer.csv`). Os dados do SEER não vão para o GitHub.
3. Ambiente com GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# O repositório não contém o CSV (os dados do SEER ficam só no Drive).
# Repositório público: a linha abaixo funciona direto.
# Repositório privado: use um token -> https://<SEU_TOKEN>@github.com/CaetanoPorto/....git
!git clone https://github.com/CaetanoPorto/tcc-sobrevivencia-cancer-mama.git repo
%cd repo

In [ ]:
# Usa o torch com GPU que já vem no Colab e instala só as bibliotecas de sobrevivência.
# (Não reinstalar torch/numpy/pandas, para não quebrar o torch com GPU do Colab.)
!pip install -q scikit-survival lifelines pycox torchtuples

In [ ]:
import os
# Ajuste para o caminho do CSV no seu Drive:
os.environ['SEER_CSV_PATH'] = '/content/drive/MyDrive/TCC/breast_cancer.csv'
assert os.path.exists(os.environ['SEER_CSV_PATH']), 'CSV não encontrado — ajuste o caminho acima'
print('CSV OK:', os.environ['SEER_CSV_PATH'])

## Passo 1 — conferir se o CSV é a mesma base do trabalho (~1 min)
O resultado esperado é **26/26 invariantes OK**. Se falhar, o CSV é diferente do usado no trabalho e não faz sentido continuar.

In [ ]:
!python scripts/profile_dataset.py

## Passo 2 — estabilidade dos modelos profundos na validação (5 sementes)
Treina DeepSurv e DeepHit na base inteira e mostra média ± desvio entre sementes **no conjunto de validação**. O teste **não** é usado aqui. Serve para ver se o C-index é estável entre sementes.

In [ ]:
!python scripts/run_deep.py --full --seeds 5

## Passo 3 — avaliação final: abre o conjunto de teste uma única vez

São duas rodadas que se complementam:
- **3a — Tabela comparativa** dos 6 modelos (incluindo o GBS) na **amostra fixa** de 100 mil, avaliada no teste dessa amostra. É a comparação principal (os mesmos dados para todos).
- **3b — Sensibilidade na base inteira** com os modelos que escalam (Cox, Cox com splines, RSF, DeepSurv, DeepHit). O **GBS é pulado automaticamente**, porque não escala para a base inteira.

Cada rodada gera: tabela (Harrell, Uno, Antolini, IBS, Brier + intervalo bootstrap), calibração por decil, métricas por era, validação temporal 2010-15 → 2016-17 e sensibilidade dos casos de tempo desconhecido.

In [ ]:
# 3a — comparação principal na amostra fixa (o GBS leva de 1 a 1,5 h; mantenha a aba aberta)
!python scripts/run_evaluation.py --sample-n 100000 --eval-set test --n-boot 200

In [ ]:
# 3b — sensibilidade na base inteira (sem GBS)
!python scripts/run_evaluation.py --full --eval-set test --n-boot 200